In [21]:
from prody import *

from matplotlib.pylab import *
ion()

import pandas as pd
import os

In [22]:
# Dictionary of PIN1, spg and FYN structures used in elastic network model

enm_ids = {"PIN1": ["2M8I", "2RUC"],
           "spg": ["1PGA", "1PGX"],
           "FYN": ["1G83"],}

enm_cutoffs = {"2M8I": "",
               "2RUC":"and resnum > 4", # 1st 4 aa not PIN1
               "1PGA": "and resnum >1",
               "1PGX": "and resnum <76",
               "1G83": ""
               }


enm_res_conversion = {"2M8I": -4,
                      "2RUC": 51,
                      "1PGA": 228,
                      "1PGX": 291,
                      "1G83": 85,}
                     

In [23]:
gene = 'FYN'

In [24]:
import scipy, prody
print(scipy.__version__, prody.__version__)



1.15.2 2.4.0


In [25]:
os.makedirs("../PDB_files", exist_ok=True)
pathPDBFolder("../PDB_files", divided=False)

In [26]:
# Use ENM (ProDy) to get the max fluctuations across the 10 fastest modes for each structure

num_modes = 10
all_gene_dfs = []

for pdb_id in enm_ids[gene]:

    # Load the structure
    structure = parsePDB(pdb_id)
    
    # Apply chain and cutoff selection
    selection_string = f"calpha and chain A {enm_cutoffs[pdb_id]}"
    calphas = structure.select(selection_string)
    print(f"{pdb_id}: selected {len(calphas)} Cα atoms with cutoff '{enm_cutoffs[pdb_id]}'")

    # Build GNM and calculate modes
    gnm = GNM(pdb_id)
    gnm.buildKirchhoff(calphas)
    n_calpha = calphas.numAtoms()
    gnm.calcModes(n_calpha - 1)  # internal modes are n-1. 1 is trivial (0, translation + rotation)

    # Collect fluctuations for the 10 fastest modes
    fluctuations_list = []
    for i in range(1, num_modes + 1):
        mode = gnm[-i]
        fluctuations = calcSqFlucts(mode)
        fluctuations_list.append(fluctuations)

    # Take the max across modes
    fluctuations_array = np.stack(fluctuations_list, axis=1)
    max_fluctuations = np.max(fluctuations_array, axis=1)

    # Convert to DataFrame
    df = pd.DataFrame({
        'Residue Index': np.arange(len(max_fluctuations)),
        'Max Fluctuation (10 fastest modes)': max_fluctuations
    })
    df['Residue'] = df['Residue Index'] + enm_res_conversion[pdb_id]
    df['PDB_ID'] = pdb_id

    all_gene_dfs.append(df)

# Concatenate all PDBs for this gene
ENM_10_df = pd.concat(all_gene_dfs, ignore_index=True)

save_dir = "../Output/ENM_dfs"
os.makedirs(save_dir, exist_ok=True)
out_file = os.path.join(save_dir, f"ENM_{gene}_10fastest.csv")
ENM_10_df.to_csv(out_file, index=False)





1G83: selected 161 Cα atoms with cutoff ''


In [27]:
# Use ENM (Prody) to get the max fluctuations for each of the fastest 30 modes of motion for each structure

num_modes = 30
all_gene_dfs = []

for pdb_id in enm_ids[gene]:

    # Load the structure
    structure = parsePDB(pdb_id)
    
    # Apply chain and cutoff selection
    selection_string = f"calpha and chain A {enm_cutoffs[pdb_id]}"
    calphas = structure.select(selection_string)

    # Build GNM and calculate modes
    gnm = GNM(pdb_id)
    gnm.buildKirchhoff(calphas)
    n_calpha = calphas.numAtoms()
    gnm.calcModes(n_calpha-1) # internal modes are n-1. 1 is trivial (0, translation + rotation)

    # Store cumulative max fluctuations
    max_fluctuations = {}

    for i in range(1, num_modes + 1):
        mode = gnm[-i]
        fluctuations = calcSqFlucts(mode)

        for idx, fluct in enumerate(fluctuations):
            if idx not in max_fluctuations:
                max_fluctuations[idx] = [0] * num_modes
            if i == 1:
                max_fluctuations[idx][i - 1] = fluct
            else:
                max_fluctuations[idx][i - 1] = max(max_fluctuations[idx][i - 2], fluct)

    # Convert to DataFrame
    df = pd.DataFrame.from_dict(
        max_fluctuations, orient='index',
        columns=[f'Max Fluctuation (-{i})' for i in range(1, num_modes + 1)]
    )
    df.insert(0, 'Residue Index', df.index)
    df['Residue'] = df['Residue Index'] + enm_res_conversion[pdb_id]
    df['PDB_ID'] = pdb_id

    all_gene_dfs.append(df)

# Concat PDBs for a gene
ENM_df = pd.concat (all_gene_dfs,ignore_index=True)

save_dir = "../Output/ENM_dfs"
out_file = os.path.join(save_dir, f"ENM_{gene}.csv")
ENM_df.to_csv(out_file, index=False)